# 02 — Dimensionality reduction on the {VISp, SCm, MRN} triplet (3-D, time-coloured)

**Cleaned 2026-07-15.** The earlier version of this notebook tested a lot of machinery
(the MSR validation gate, the rate-matched control, task-selectivity, held-out decoding,
a 7-method 2-D zoo, choice-coloured maps). **Those results are recorded** in
`docs/worklog/2026-07-13_msr-and-dimred.md` and
`docs/knowledge/where_we_are_HMM_rSLDS_dimred.md` §3.6, and the standing conclusion —
**MSR is not adopted as a unit-selection method** — has not changed. So none of that is
re-run here.

What this notebook now does, and only this:

1. Eats the **unified 40 ms preprocessing** (`data/processed/…bin40…npz`) — the same frozen
   tensor every model in the project consumes, config stamped inside. No ad-hoc binning.
2. Keeps MSR **only** to define one comparison unit set (top-half by MSR), so the *effect*
   of that selection on the geometry is visible next to the full population.
3. Shows the four embeddings that were the most informative — **PCA, dPCA, MDS, Laplacian
   eigenmaps — in interactive 3-D** (Plotly — **drag any panel to rotate**, scroll to zoom),
   coloured **by time**, for **all units** and the **MSR half**, in a **post-stimulus** view
   and a **full-window (with pre-stimulus)** view.
4. **Merges the scientific results from notebook 04** (§6–9) — dPCA variance split, GPFA
   timescales, RRR communication subspace, and the joint cross-area manifold — **re-run on
   this same 40 ms tensor**, so the whole dimensionality-reduction story is in one notebook
   eating one preprocessing.

No choice-coloured plots (that split is in the worklog). §1–5 are *illustration* (fitted and
shown on the same trials); §6–9 are the *results* (a number with a control or a held-out test).

> **Interactive 3-D:** every 3-D figure is a Plotly scene — rotate by dragging, zoom by
> scrolling. Each is also written to `results/figures/phase3_dimred/*.html`, openable in a
> browser. (The 2-D bar/line charts in §6 and §8 stay static matplotlib.) If a panel shows
> blank when you open the notebook, run the cell — VSCode renders the Plotly scene live.

In [ ]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
# Store the plotly mimetype so the interactive scene renders when the notebook is opened in
# VSCode (offline) AND carry a CDN reference so it also renders in classic Jupyter/nbviewer.
pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.viz.style import set_style, savefig, figure_path, region_color, OKABE_ITO, SEQUENTIAL
set_style()

# --- CVD-safe plotly colourscales (match the matplotlib cividis / RdBu_r used elsewhere) ---
PLOTLY_SEQ = "Cividis"      # time, sequential
PLOTLY_DIV = "RdBu"         # + reversescale=True -> blue=negative(baseline), red=positive(evoked)
PLOTLY_MAX = 1000           # markers shown per 3-D panel (keeps rotation smooth + notebook small)
CAMERA = dict(eye=dict(x=1.6, y=1.5, z=1.0))
BLANK_AX = dict(showticklabels=False, title="", showspikes=False)

# --- the frozen 40 ms pipeline output (src/data/preprocess.py); NOT rebuilt here ---
PROCESSED = ROOT / "data" / "processed" / "5c0c560e__stimOn_pre400_post800_bin40_rate1.npz"
SESSION_EID = "5c0c560e-9e1f-45e9-b66e-e4ee7855be84"
REGIONS  = ["VISp", "SCm", "MRN"]
SMOOTH_MS = 40.0                 # the project's standing geometry smoothing (sigma, in ms)
SEED = 0
PHASE = "phase3_dimred"

# 3-D embeddings are O(n^2) (MDS) or nearly so; show every method on the same subsample.
N_SUB, N_SUB_MDS = 1800, 700
rng = np.random.default_rng(SEED)
OUT = ROOT / "results" / "dimred"; OUT.mkdir(parents=True, exist_ok=True)
print(f"figures -> results/figures/{PHASE}/   tables -> results/dimred/")

## 1. Load the frozen 40 ms tensor, and the spike trains MSR needs

`load_processed` returns exactly what the pipeline froze: the `(trials, bins, units)` count
tensor per region (units already filtered to ≥ 1 Hz), the bin centres, the pre-stimulus
`baseline_mask`, and the trial covariates — **config stamped inside the file**. This is the
"pass the data through the preprocessing pipeline at 40 ms" step, done once, upstream.

MSR needs each unit's **continuous** spike train (it sweeps bin widths far wider than a trial
window), which is not in the frozen tensor — so the raw session is loaded once for that. The
pipeline records `keep_idx` (which of a region's original units survived the rate cut), and we
use it to line the trains up with the tensor columns, then **assert** the alignment by
re-binning. A mismatch here would silently attach the wrong MSR score to a unit.

In [ ]:
from src.data.preprocess import load_processed, load_session, align_times, PreprocessConfig
from src.data.build_matrices import region_spike_trains
from src.selection.msr import msr_table

data = load_processed(PROCESSED)
cfg_d = data["config"]
BIN_S, PRE_S, POST_S = cfg_d["bin_size_s"], cfg_d["pre_s"], cfg_d["post_s"]
bin_centers = data["bin_centers"]
n_trials = data["n_trials"]
print(f"loaded {PROCESSED.name}")
print(f"  {n_trials} trials | {BIN_S*1000:.0f} ms bins | window [-{PRE_S}, +{POST_S}] s | "
      f"{int((bin_centers<0).sum())} pre-stimulus bins")
print("  units:", {r: data['regions'][r]['counts'].shape[2] for r in REGIONS})

# raw load, once, ONLY for the continuous spike trains MSR consumes
cfg = PreprocessConfig(align_event=cfg_d["align_event"], pre_s=PRE_S, post_s=POST_S,
                       bin_size_s=BIN_S, min_rate_hz=cfg_d["min_rate_hz"],
                       min_rt_s=cfg_d["min_rt_s"], max_rt_s=cfg_d["max_rt_s"],
                       regions=tuple(cfg_d["regions"]), qc=cfg_d["qc"])
one = None
from src.data.access import get_one
one = get_one(cache_dir=ROOT / "data" / "ONE")
trials, mask, probe_loads = load_session(one, SESSION_EID, cfg=cfg)
events = align_times(trials, mask, cfg.align_event)
assert len(events) == n_trials, f"{len(events)} events vs {n_trials} tensor trials"
TASK = (float(events.min() - PRE_S - 1), float(events.max() + POST_S + 1))
print(f"raw session loaded for spike trains ({len(events)} events)")

In [ ]:
# MSR per region, on the *kept* units, aligned to the tensor columns via keep_idx
msr_top, trains_by_region = {}, {}
for region in REGIONS:
    counts = data["regions"][region]["counts"]
    keep_idx = data["regions"][region]["keep_idx"]
    trains_full = region_spike_trains(probe_loads, region, t_start=TASK[0], t_stop=TASK[1])
    trains = {c: trains_full[int(keep_idx[c])] for c in range(len(keep_idx))}
    trains_by_region[region] = trains        # kept for GPFA below (§7)

    # alignment assert: re-bin each train in-window, require it to equal the tensor column
    tot_tensor = counts.sum(axis=(0, 1))
    tot_train = np.array([int((np.searchsorted(trains[c], events + POST_S, "right")
                               - np.searchsorted(trains[c], events - PRE_S, "left")).sum())
                          for c in range(len(trains))])
    assert np.array_equal(tot_tensor, tot_train), f"{region}: MSR trains misaligned to tensor"

    tab = msr_table(trains, t_start=TASK[0], t_stop=TASK[1]).dropna(subset=["msr"])
    n_keep = len(tab) // 2
    msr_top[region] = np.sort(tab.head(n_keep)["unit"].to_numpy())
    print(f"[{region}] {counts.shape[2]} units, alignment VERIFIED | "
          f"MSR-top half = {n_keep} units (rate {tab.head(n_keep)['rate_hz'].median():.1f} Hz "
          f"vs {tab.tail(len(tab)-n_keep)['rate_hz'].median():.1f} Hz for the rest)")

print("\nReminder: MSR is NOT adopted as a selection method (worklog 2026-07-13). It correlates")
print("with firing rate, so 'msr_top' is also 'the louder half' — kept here only to SHOW that")
print("effect on the geometry, not as a recommended population.")

UNIT_SETS = ["all", "msr_top"]
def units_of(region, uset):
    return None if uset == "all" else msr_top[region]   # None => all columns

## 2. Build the geometry point clouds, and run the embeddings

A point is one **(trial, 20 ms→40 ms bin) population vector**. Preprocessing for geometry
(never for the HMM): `sqrt` (variance-stabilises Poisson counts), Gaussian smoothing at
**sigma = 40 ms**, z-score per unit. Then three point-cloud embeddings to **3 components**:

* **PCA** — linear, distance-preserving; variance explained is printed on the panel.
* **MDS** — preserves global distances (the honest one for "is there a trajectory").
* **Laplacian eigenmaps** — neighbourhood-graph spectral embedding.

Each embedding is **fit on the full window** (pre- and post-stimulus bins together), so the
two views below — post-only and with-prestimulus — are the *same manifold*, just showing
different bins. That is what lets you see whether the population **leaves** its baseline
region after the stimulus, rather than comparing two separately-fitted pictures.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, SpectralEmbedding
from src.geometry.embed import population_matrix

time_of_row = np.tile(bin_centers, n_trials)

def embed_3d(X, seed=SEED):
    sub = rng.choice(X.shape[0], min(N_SUB, X.shape[0]), replace=False)
    sub_mds = sub[:N_SUB_MDS]
    out = {}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        p = PCA(n_components=3, random_state=seed).fit(X)
        out["PCA"] = (p.transform(X[sub]), p.explained_variance_ratio_[:3], sub)
        out["MDS"] = (MDS(n_components=3, n_init=1, max_iter=120, random_state=seed,
                          n_jobs=-1).fit_transform(X[sub_mds]), None, sub_mds)
        out["Laplacian"] = (SpectralEmbedding(n_components=3, n_neighbors=15,
                            random_state=seed).fit_transform(X[sub]), None, sub)
    return out

METHODS = ["PCA", "MDS", "Laplacian"]
t0 = time.time()
emb = {}
for region in REGIONS:
    counts = data["regions"][region]["counts"]
    for uset in UNIT_SETS:
        X = population_matrix(counts, units_of(region, uset),
                              bin_size_s=BIN_S, smooth_sigma_ms=SMOOTH_MS)
        emb[(region, uset)] = embed_3d(X)
    print(f"[{region}] embedded  ({time.time()-t0:.0f}s)")
print(f"\nall 3-D embeddings: {time.time()-t0:.0f} s")

## 3. Post-stimulus trajectories, coloured by time

Cividis = time from stimulus onset (0 → +0.8 s), CVD-safe. Rows are **all units** vs the
**MSR half**, columns are the three methods, for each region.

What to look for: a smooth, ordered colour gradient means the population passes through a
stereotyped sequence of states (the metastable-sequence geometry the project predicts); a
colour-scrambled blob means no such trajectory at this smoothing. Compare the two rows — if
the MSR half reproduces the same path, the discarded units were geometrically redundant.

In [ ]:
def _panel_title(region, uset, m, evr):
    n = data["regions"][region]["counts"].shape[2] if uset == "all" else len(msr_top[region])
    head = m if m != "PCA" else f"PCA {100*evr[0]:.0f}/{100*evr[1]:.0f}/{100*evr[2]:.0f}%"
    return f"{head} · {uset} ({n}u)"

def cloud_grid(region, *, diverging):
    # 2 (unit set) x 3 (method) grid of rotatable 3-D scatters, coloured by time.
    # diverging=False -> post-stimulus bins only, Cividis (0 -> +0.8 s).
    # diverging=True  -> full window, RdBu centred at onset (blue=baseline, red=evoked).
    titles = [_panel_title(region, u, m, emb[(region, u)][m][1])
              for u in UNIT_SETS for m in METHODS]
    fig = make_subplots(rows=len(UNIT_SETS), cols=len(METHODS),
                        specs=[[{"type": "scene"}]*len(METHODS) for _ in UNIT_SETS],
                        subplot_titles=titles, horizontal_spacing=0.01, vertical_spacing=0.07)
    tmax = float(max(-bin_centers.min(), bin_centers.max()))
    for i, uset in enumerate(UNIT_SETS):
        for j, m in enumerate(METHODS):
            Y, evr, idx = emb[(region, uset)][m]
            t = time_of_row[idx]
            if not diverging:
                keep = t >= 0
                Y, t = Y[keep], t[keep]
            if Y.shape[0] > PLOTLY_MAX:
                s = rng.choice(Y.shape[0], PLOTLY_MAX, replace=False); Y, t = Y[s], t[s]
            first = (i == 0 and j == 0)
            fig.add_trace(go.Scatter3d(
                x=Y[:, 0], y=Y[:, 1], z=Y[:, 2], mode="markers", showlegend=False,
                hovertemplate="t=%{marker.color:.2f}s<extra></extra>",
                marker=dict(size=2.0, color=t, colorscale=PLOTLY_DIV if diverging else PLOTLY_SEQ,
                            reversescale=diverging, cmin=(-tmax if diverging else 0.0),
                            cmax=tmax, cmid=(0.0 if diverging else None), opacity=0.8,
                            showscale=first,
                            colorbar=dict(title="time (s)", x=1.01, len=0.7) if first else None)),
                row=i+1, col=j+1)
    fig.update_scenes(xaxis=BLANK_AX, yaxis=BLANK_AX, zaxis=BLANK_AX, camera=CAMERA)
    win = "full window (blue=baseline, red=evoked)" if diverging else "post-stimulus"
    fig.update_layout(height=760, width=1150, margin=dict(l=0, r=60, t=64, b=0),
                      title=dict(text=f"{region} — {win} · 3-D, DRAG TO ROTATE", x=0.5))
    return fig

for region in REGIONS:
    fig = cloud_grid(region, diverging=False)
    fig.write_html(str(figure_path(PHASE, f"3d_post_{region}", ext="html")),
                   include_plotlyjs="cdn")
    fig.show()

## 4. The same manifolds **with the pre-stimulus baseline included**

Same embeddings, now showing every bin. Colour is a **diverging** map centred at stimulus
onset (`RdBu_r`, CVD-safe): **blue = before the stimulus**, **red = after**. So the
pre-stimulus baseline is not just present but *visually separated* from the evoked response.

This is the view that answers "does the population **leave** its resting state?" — if the blue
(baseline) points sit in a tight clump and the red points fan out along a path, the trajectory
departs baseline at stimulus onset. If blue and red are interleaved, they do not.

In [ ]:
for region in REGIONS:
    fig = cloud_grid(region, diverging=True)
    fig.write_html(str(figure_path(PHASE, f"3d_prestim_{region}", ext="html")),
                   include_plotlyjs="cdn")
    fig.show()

## 5. dPCA in 3-D — the demixed axes, coloured by time

The other three methods do not know *why* the trajectory has the shape it has. dPCA
(Kobak et al. 2016) does: it finds axes that isolate **time** (condition-independent),
**stimulus**, and **decision** variance. Plotting the leading component of each as one 3-D
space is the natural dPCA figure — and, unlike a point cloud, its axes *mean* something.

Here the four **stimulus × decision condition means** are drawn as trajectories over the full
window, coloured by time. Read it by axis, not by colour:

* movement along the **time axis** = the condition-independent sweep every trial shares;
* separation along the **stimulus axis** = when stimulus side becomes encoded;
* separation along the **decision axis** = when choice becomes encoded.

dPCA is a **trial-averaged** method and is limited by trials, not units, so it is shown on
the **full population only** (halving the units would just add noise to the 25-trial decision
cells — see the worklog). The marker shape distinguishes the four conditions; colour is time.

The condition means rest on only **K = 25 trials each**, so the raw trajectories are jittery;
they are **lightly Gaussian-smoothed along time for display only** (the variance percentages
in the axis labels come from the *unsmoothed* fit). Read the geometry, not the wiggle.

In [ ]:
from scipy.ndimage import gaussian_filter1d
from src.dynamics.dpca import balanced_conditions, fit_dpca, variance_by_marginalization

stim_right = ~np.isnan(data["trials"]["contrastRight"])
choice = data["trials"]["choice"]
COND_SYM = {(0, 0): "circle", (0, 1): "diamond", (1, 0): "square", (1, 1): "cross"}
COND_LAB = {(0, 0): "stim L / chose L", (0, 1): "stim L / chose R",
            (1, 0): "stim R / chose L", (1, 1): "stim R / chose R"}
_smooth = lambda v: gaussian_filter1d(v, sigma=1.0, mode="nearest")   # display only

# fit dPCA once per region; keep the variance tables + the demixed components
dpca_var, dpca_Z, dpca_K = {}, {}, {}
for region in REGIONS:
    X, trialX, K = balanced_conditions(data["regions"][region]["counts"], stim_right, choice,
                                       seed=SEED)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model, Z = fit_dpca(X, trialX, n_components=3)
    dpca_var[region], dpca_Z[region], dpca_K[region] = variance_by_marginalization(model), Z, K

fig = make_subplots(rows=1, cols=len(REGIONS),
                    specs=[[{"type": "scene"}]*len(REGIONS)],
                    subplot_titles=[f"{r}  (K={dpca_K[r]}/cell)" for r in REGIONS])
for j, region in enumerate(REGIONS):
    Z = dpca_Z[region]
    for s in (0, 1):
        for d in (0, 1):
            xs, ys, zs = (_smooth(Z["t"][0, s, d]), _smooth(Z["s"][0, s, d]),
                          _smooth(Z["d"][0, s, d]))
            first = (j == 0 and s == 0 and d == 0)
            fig.add_trace(go.Scatter3d(
                x=xs, y=ys, z=zs, mode="lines+markers", showlegend=False,
                name=COND_LAB[(s, d)], line=dict(color="lightgray", width=2),
                marker=dict(size=3, symbol=COND_SYM[(s, d)], color=bin_centers,
                            colorscale=PLOTLY_SEQ, showscale=first,
                            colorbar=dict(title="time (s)", x=1.01, len=0.7) if first else None)),
                row=1, col=j+1)
    var = dpca_var[region].set_index("marginalization")["pct_of_explained"]
    sc = fig.layout["scene" if j == 0 else f"scene{j+1}"]
    sc.xaxis.title.text = f"time ({var.get('t',0):.0f}%)"
    sc.yaxis.title.text = f"stim ({var.get('s',0):.0f}%)"
    sc.zaxis.title.text = f"decision ({var.get('d',0):.0f}%)"
    sc.camera = CAMERA
fig.update_layout(height=520, width=1250, margin=dict(l=0, r=60, t=70, b=0),
                  title=dict(x=0.5, text="dPCA demixed 3-D · symbol = condition (o/♦/□/+ = "
                             "LL/LR/RL/RR), colour = time · DRAG TO ROTATE"))
fig.write_html(str(figure_path(PHASE, "3d_dpca_demixed", ext="html")), include_plotlyjs="cdn")
fig.show()

---
# The scientific results (merged from notebook 04, now on the 40 ms pipeline)

Everything above is *illustration* — fitted and shown on the same trials. Everything below
is a **result**: a number with a control or a held-out test. These were developed in
`04_manifold_dpca_gpfa_rrr.ipynb` (which built its own 50 ms tensors); they are re-run here on
the **same frozen 40 ms tensor** as the plots above, so the whole dim-red story now lives in
one notebook eating one preprocessing. Full method notes:
`docs/knowledge/dPCA_maths_and_the_attractor_sequence.md` and
`docs/knowledge/where_we_are_HMM_rSLDS_dimred.md` §3.

## 6. dPCA variance split — the project's core prediction, as a number

The 3-D plot above showed the demixed *axes*; this is the quantity that matters. dPCA
partitions the population variance by **cause** — condition-independent *time*, *stimulus*,
*decision*, and interactions. The prediction: VISp carries relatively more **stimulus**
variance, SCm/MRN more **decision** variance. `decision − stimulus` should therefore climb
along the visual→motor axis. ("stimulus %" aggregates the `s`, `st`, `ts` marginalisations;
"decision %" the `d`, `dt`, `td` ones — see the maths doc.)

In [ ]:
from src.dynamics.dpca import MARGINALIZATION_NAMES

keys = sorted({k for r in REGIONS for k in dpca_var[r]["marginalization"]})
fig, (axb, axg) = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [2, 1]})

w = 0.25
for i, region in enumerate(REGIONS):
    d = dpca_var[region].set_index("marginalization")["pct_of_explained"]
    axb.bar(np.arange(len(keys)) + (i-1)*w, [d.get(k, 0.0) for k in keys], w,
            color=region_color(region), edgecolor="black", linewidth=0.5, label=region)
axb.set(xticks=range(len(keys)),
        xticklabels=[MARGINALIZATION_NAMES.get(k, k).replace(" (condition-independent)", "")
                     for k in keys],
        ylabel="% of explained variance", title="dPCA: what CAUSES the population variance?")
axb.tick_params(axis="x", rotation=25); axb.legend(frameon=False)

comp = []
for region in REGIONS:
    d = dpca_var[region].set_index("marginalization")["pct_of_explained"]
    stim = sum(d.get(k, 0) for k in ["s", "st", "ts"])
    dec = sum(d.get(k, 0) for k in ["d", "dt", "td"])
    comp.append({"region": region, "stimulus_%": round(stim, 1), "decision_%": round(dec, 1),
                 "time_%": round(d.get("t", 0), 1), "decision_minus_stimulus": round(dec-stim, 1)})
comp = pd.DataFrame(comp)

axg.bar(comp["region"], comp["decision_minus_stimulus"],
        color=[region_color(r) for r in comp["region"]], edgecolor="black")
axg.axhline(0, color="black", lw=1)
axg.set(ylabel="decision − stimulus  (% variance)", title="the gradient")
fig.suptitle("dPCA variance by cause (40 ms) — decision−stimulus should rise VISp→SCm→MRN",
             fontweight="bold")
fig.tight_layout(); savefig(fig, PHASE, "dpca_variance_split"); plt.show()

comp.to_csv(OUT / "dpca_variance.csv", index=False)
print(comp.to_string(index=False))
print("\ntime_% dominates everywhere — most of any raw PCA trajectory is this shared temporal")
print("sweep, in EVERY area. That shared sweep IS the sequence backbone; the decision share is")
print("what the sequence ENCODES. See dPCA_maths_and_the_attractor_sequence.md.")

## 7. GPFA — the dynamics are slower than any bin, and the smoothing is *inferred*

Every embedding above ran on data **we** smoothed. GPFA (Yu et al. 2009) does not: it models
each latent as a Gaussian process across time and the spikes as noisy observations, so it takes
**raw counts** and *infers* the timescale `tau` per dimension. That makes it the honest check on
whether our trajectories are our smoother's doing — and the fitted `tau` is itself a result: the
timescale the population dynamics actually run at.

In [ ]:
from src.dynamics.gpfa import trials_to_spiketrains, fit_gpfa, latent_timescales

X_DIM = 5
t0 = time.time()
gpfa_lat, gpfa_tau = {}, {}
for region in REGIONS:
    st = trials_to_spiketrains(trains_by_region[region], events, pre_s=PRE_S, post_s=POST_S)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model, lat = fit_gpfa(st, bin_size_s=BIN_S, x_dim=X_DIM, em_max_iters=50)
    gpfa_lat[region], gpfa_tau[region] = lat, latent_timescales(model, BIN_S)
    print(f"[{region}] latents {lat.shape} | tau (ms): {np.round(np.sort(gpfa_tau[region])[::-1],0)}")

tau_rows = []
for r in REGIONS:
    row = {"region": r}
    for i, t in enumerate(np.sort(gpfa_tau[r])[::-1]):
        row[f"tau{i+1}_ms"] = round(float(t), 0)
    tau_rows.append(row)
tau_tab = pd.DataFrame(tau_rows); tau_tab.to_csv(OUT / "gpfa_timescales.csv", index=False)
print(f"\nGPFA: {time.time()-t0:.0f} s. Slowest tau by region:",
      {r: float(np.round(gpfa_tau[r].max(), 0)) for r in REGIONS})

In [ ]:
# GPFA latents 1-3 in 3-D (rotatable): faint single trials + the trial-mean coloured by time
n_show = 25
fig = make_subplots(rows=1, cols=len(REGIONS), specs=[[{"type": "scene"}]*len(REGIONS)],
                    subplot_titles=[f"{r} (slowest tau {gpfa_tau[r].max():.0f} ms)" for r in REGIONS])
for j, region in enumerate(REGIONS):
    lat = gpfa_lat[region]
    for i in rng.choice(lat.shape[0], min(n_show, lat.shape[0]), replace=False):
        fig.add_trace(go.Scatter3d(x=lat[i, 0], y=lat[i, 1], z=lat[i, 2], mode="lines",
                                   line=dict(color=region_color(region), width=1), opacity=0.18,
                                   showlegend=False, hoverinfo="skip"), row=1, col=j+1)
    m = lat.mean(0); first = (j == 0)
    fig.add_trace(go.Scatter3d(x=m[0], y=m[1], z=m[2], mode="markers", showlegend=False,
                  marker=dict(size=4, color=bin_centers, colorscale=PLOTLY_SEQ, showscale=first,
                              line=dict(color="black", width=1),
                              colorbar=dict(title="time (s)", x=1.01, len=0.7) if first else None)),
                  row=1, col=j+1)
fig.update_scenes(xaxis=BLANK_AX, yaxis=BLANK_AX, zaxis=BLANK_AX, camera=CAMERA)
fig.update_layout(height=520, width=1250, margin=dict(l=0, r=60, t=70, b=0),
                  title=dict(x=0.5, text="GPFA latents 1–3 · inferred smoothness on RAW counts "
                             "(faint = single trials, dots = trial mean) · DRAG TO ROTATE"))
fig.write_html(str(figure_path(PHASE, "gpfa_latents_3d", ext="html")), include_plotlyjs="cdn")
fig.show()
print("Every tau is far above the 40 ms bin, so the bin loses nothing, and the smoothed")
print("point clouds above are not an artefact of the smoother. Motor areas run slowest.")

## 8. RRR — the communication subspace between areas (the common-drive result)

Reduced-rank regression (Semedo et al. 2019): predict area B's population from area A's,
restricted to rank `r`. The rank at which held-out R² **saturates** is the dimensionality of
the channel through which A influences B. The reference is the **within-area control** — split
the target's own units in half and predict one half from the other — because *any* two
populations correlate. CV is grouped **by trial**. A cross-area rank *below* the within-area
rank means a genuinely restricted channel; *equal* means the areas share whatever is there.

In [ ]:
from src.dynamics.rrr import rrr_cv_curve, within_area_control, saturation_rank

mats = {r: population_matrix(data["regions"][r]["counts"], None,
                             bin_size_s=BIN_S, smooth_sigma_ms=SMOOTH_MS) for r in REGIONS}
n_bins = data["regions"][REGIONS[0]]["counts"].shape[1]
groups = np.repeat(np.arange(n_trials), n_bins)
PAIRS = [("VISp", "SCm"), ("VISp", "MRN"), ("SCm", "MRN"), ("MRN", "SCm")]
RANKS = list(range(1, 11))

t0 = time.time()
curves, rrr_rows = {}, []
for src, tgt in PAIRS:
    c = rrr_cv_curve(mats[src], mats[tgt], groups, ranks=RANKS); curves[(src, tgt)] = c
    rrr_rows.append({"source": src, "target": tgt, "kind": "cross-area",
                     "best_r2": round(float(c["mean_r2"].max()), 3), "sat_rank": saturation_rank(c)})
for tgt in REGIONS:
    c = within_area_control(mats[tgt], groups, ranks=RANKS, seed=SEED); curves[(tgt, tgt)] = c
    rrr_rows.append({"source": f"{tgt} (half)", "target": tgt, "kind": "within-area control",
                     "best_r2": round(float(c["mean_r2"].max()), 3), "sat_rank": saturation_rank(c)})
rrr_tab = pd.DataFrame(rrr_rows); rrr_tab.to_csv(OUT / "rrr_ranks.csv", index=False)
print(rrr_tab.to_string(index=False)); print(f"\nRRR: {time.time()-t0:.0f} s")

fig, axes = plt.subplots(1, len(REGIONS), figsize=(5.0*len(REGIONS), 4), sharey=True)
for ax, tgt in zip(axes, REGIONS):
    for src, t in [pr for pr in PAIRS if pr[1] == tgt]:
        c = curves[(src, tgt)]
        ax.errorbar(c["rank"], c["mean_r2"], yerr=c["sem_r2"], marker="o", ms=4, lw=1.6,
                    color=region_color(src), label=f"{src}->{tgt}")
    c = curves[(tgt, tgt)]
    ax.errorbar(c["rank"], c["mean_r2"], yerr=c["sem_r2"], marker="s", ms=4, lw=1.8,
                color="black", ls="--", label=f"{tgt} half->half (control)")
    ax.set(xlabel="rank", title=f"target: {tgt}"); ax.legend(frameon=False, fontsize=8)
axes[0].set_ylabel("held-out R2 (grouped by trial)")
fig.suptitle("Communication subspace — read cross-area vs the BLACK within-area control",
             fontweight="bold")
fig.tight_layout(); savefig(fig, PHASE, "rrr_communication"); plt.show()

## 9. One manifold, three areas — the joint space and its cross-area covariance

The areas are recorded **simultaneously**, so they share the (trial, bin) samples. Concatenate
all units, take the **joint PCA**; because PCA is linear the joint trajectory decomposes
*exactly* into per-area contributions `Y = Σ_a Y_a` (asserted in `src/geometry/joint.py`), so
the three areas can be plotted on one set of axes. An area that *drives* the shared trajectory
has a large, time-structured contribution; the variance shares do **not** sum to 1 — the
remainder is **cross-area covariance**, the shared signal itself.

In [ ]:
from src.geometry.joint import joint_matrix, joint_latent, partial_projections, area_drive

Xj, slices = joint_matrix(mats)
pca_j, Yj = joint_latent(Xj, n_components=10, seed=SEED)
parts = partial_projections(Xj, pca_j, slices)                 # asserts sum(parts) == Yj
drive = area_drive(parts); drive["n_units"] = [mats[r].shape[1] for r in drive["region"]]
drive.to_csv(OUT / "joint_area_drive.csv", index=False)

v = pca_j.explained_variance_ratio_
print(f"joint: {Xj.shape[1]} units, {Xj.shape[0]:,} samples | "
      f"PC1 {100*v[0]:.1f}% / PC2 {100*v[1]:.1f}% / PC1-10 {100*v[:10].sum():.1f}%")
print(drive.to_string(index=False))
print(f"var shares sum to {drive['var_share'].sum():.3f} -> "
      f"cross-area covariance ~= {1 - drive['var_share'].sum():.2f} (the shared signal)")

SHAPES = {"VISp": "circle", "SCm": "diamond", "MRN": "square"}
t_row = np.tile(bin_centers, n_trials)
pick = rng.choice(Yj.shape[0], min(PLOTLY_MAX, Yj.shape[0]), replace=False)
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]],
                    subplot_titles=["each area's contribution · colour = area",
                                    "shape = area · colour = time"])
for region in REGIONS:
    Y = parts[region]
    fig.add_trace(go.Scatter3d(x=Y[pick, 0], y=Y[pick, 1], z=Y[pick, 2], mode="markers",
                  name=f"{region} ({mats[region].shape[1]}u)",
                  marker=dict(size=2.0, color=region_color(region), opacity=0.5)), row=1, col=1)
for k, region in enumerate(REGIONS):
    Y = parts[region]; first = (k == 0)
    fig.add_trace(go.Scatter3d(x=Y[pick, 0], y=Y[pick, 1], z=Y[pick, 2], mode="markers",
                  showlegend=False, marker=dict(size=2.4, symbol=SHAPES[region], color=t_row[pick],
                  colorscale=PLOTLY_SEQ, showscale=first,
                  colorbar=dict(title="time (s)", x=1.01, len=0.7) if first else None)),
                  row=1, col=2)
fig.update_scenes(xaxis=BLANK_AX, yaxis=BLANK_AX, zaxis=BLANK_AX, camera=CAMERA)
fig.update_layout(height=560, width=1200, margin=dict(l=0, r=60, t=70, b=0),
                  legend=dict(x=0.0, y=0.98),
                  title=dict(x=0.5, text=f"Common manifold across the triplet "
                             f"(joint PCA of {Xj.shape[1]} units) · DRAG TO ROTATE"))
fig.write_html(str(figure_path(PHASE, "joint_manifold_3d", ext="html")), include_plotlyjs="cdn")
fig.show()

In [ ]:
lines = [
    "# 02 — dimensionality reduction + merged 04 results (40 ms pipeline)",
    "",
    f"Session `{SESSION_EID}`, **40 ms unified pipeline** (`{PROCESSED.name}`), "
    f"{n_trials} trials, window [-{PRE_S}, +{POST_S}] s, geometry smoothing sigma {SMOOTH_MS:.0f} ms.",
    "Units (>= 1 Hz): " + ", ".join(f"{r} {data['regions'][r]['counts'].shape[2]}" for r in REGIONS)
    + "; MSR-top half: " + ", ".join(f"{r} {len(msr_top[r])}" for r in REGIONS) + ".",
    "",
    "## Visualisation (interactive 3-D Plotly, coloured by time — *.html, drag to rotate)",
    "- `3d_post_<region>.html` / `3d_prestim_<region>.html` — PCA/MDS/Laplacian, post-stim +",
    "  full window (diverging: blue=baseline, red=evoked). all units vs MSR-top half.",
    "- `3d_dpca_demixed.html` — dPCA time/stimulus/decision axes, colour=time, shape=condition.",
    "- `gpfa_latents_3d.html`, `joint_manifold_3d.html` — the GPFA + joint-manifold scenes.",
    "  (2-D: `dpca_variance_split.png`, `rrr_communication.png` stay static matplotlib.)",
    "",
    "## dPCA variance by cause (the core prediction)",
    "",
    comp.to_markdown(index=False),
    "",
    "## GPFA fitted timescales (ms)",
    "",
    tau_tab.to_markdown(index=False),
    "",
    "## RRR communication subspace (vs within-area control)",
    "",
    rrr_tab.to_markdown(index=False),
    "",
    "## Joint manifold — each area's share of the shared latent",
    "",
    drive.to_markdown(index=False),
    f"\nvar shares sum to {drive['var_share'].sum():.3f}; the remaining "
    f"~{1-drive['var_share'].sum():.2f} is cross-area covariance.",
    "",
    "MSR is kept only as a comparison set, NOT adopted (worklog 2026-07-13). Maths of the dPCA",
    "result and its link to the attractor sequence: `docs/knowledge/dPCA_maths_and_the_attractor_sequence.md`.",
]
(OUT / "summary_02.md").write_text("\n".join(lines), encoding="utf-8")
print((OUT / "summary_02.md").read_text(encoding="utf-8"))